# Notebook 06: DuckDB + Parquet Querying

**Phase 5 — DuckDB: Parquet as a First-Class Data Format**

Parquet is a columnar binary format designed for analytics.  DuckDB treats
Parquet files as queryable tables with no intermediate loading step — the file
*is* the table.

| Feature | What it means |
|:---|:---|
| Schema inference | Column names and types read from the Parquet footer — no `CREATE TABLE` |
| Glob patterns | `'orders_*.parquet'` queries many files as one virtual table |
| Hive partitioning | Directory names (`year=1996/`) become filter-able columns |
| Min/max pruning | Row groups outside the filter range are skipped — no I/O |
| `COPY ... TO` | Write any query result back to Parquet in one statement |

---

## Prerequisites

Run `scripts/generate_data.py` first to populate `data/parquet/`.


In [ ]:
import pathlib
import sys

import duckdb
import pandas as pd

ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PARQUET_DIR = ROOT / "data" / "parquet"

duck = duckdb.connect()

SQL_DIR = ROOT / "sql" / "parquet"


def load_section(filename: str, section: str) -> str:
    text = (SQL_DIR / filename).read_text(encoding="utf-8")
    blocks: dict[str, str] = {}
    current: str | None = None
    acc: list[str] = []
    for line in text.splitlines():
        if line.startswith("-- §"):
            if current is not None:
                blocks[current] = "\n".join(acc).strip()
            current = line[4:].strip()
            acc = []
        else:
            acc.append(line)
    if current is not None:
        blocks[current] = "\n".join(acc).strip()
    if section not in blocks:
        raise KeyError(f"Section '{section}' not found in {filename}. Available: {list(blocks)}")
    return blocks[section]


print(f"DuckDB {duckdb.__version__}")
print(f"Parquet dir: {PARQUET_DIR}")


---

## 1 · Direct File Query and Schema Inference

DuckDB can query a Parquet file directly with `read_parquet()` — no `LOAD`,
no `CREATE TABLE`, no migration step.  It reads the column names, types, and
statistics from the **Parquet footer** before reading a single data byte.

This means you can explore an unfamiliar Parquet file with `DESCRIBE` in
milliseconds, without knowing its schema ahead of time.


In [ ]:
# Schema inference: see what columns and types the Parquet file declares.
# DuckDB reads only the footer — this completes in < 1ms for any size file.
print(load_section("01_reading_parquet.sql", "schema_inference"))
print()
duck.execute(
    load_section("01_reading_parquet.sql", "schema_inference")
          .replace("'data/", f"'{PARQUET_DIR}/")
          .replace(".parquet'", ".parquet'")
).df()


In [ ]:
# TPC-H Q1 pricing summary — pure aggregation on 6M lineitem rows.
# No connection string, no server, no CREATE TABLE — just the file path.
sql = load_section("01_reading_parquet.sql", "direct_query")
# Substitute runtime path
sql = sql.replace("'data/parquet/lineitem.parquet'",
                  f"'{PARQUET_DIR}/lineitem.parquet'")
print(sql)
print()
duck.execute(sql).df()


---

## 2 · Glob Patterns

A glob pattern like `'orders_partitioned/**/*.parquet'` lets DuckDB read
**many files as a single virtual table** — the schemas are unified
automatically.

This is the pattern used when data is written in time-based batches
(one file per day, per hour, etc.) and you want to query across all of them
without manually listing file paths.


In [ ]:
sql = load_section("01_reading_parquet.sql", "glob_query")
sql = sql.replace("'data/parquet/", f"'{PARQUET_DIR}/")
print(sql)
print()
duck.execute(sql).df()


---

## 3 · Hive Partitioning

**Hive partitioning** encodes a column value in the directory name:

```
orders_partitioned/
  year=1992/data.parquet
  year=1993/data.parquet
  ...
  year=1998/data.parquet
```

When `hive_partitioning = true`, DuckDB:
1. Extracts `year` as a virtual column from the path.
2. Skips entire directories when a `WHERE year = ...` filter is present —
   **no files are opened** for excluded partitions.

This is **partition pruning** — a coarser, file-level skip that operates
before min/max row group pruning.


In [ ]:
sql = load_section("02_partitioning_and_pushdown.sql", "hive_partition_query")
sql = sql.replace("'data/parquet/", f"'{PARQUET_DIR}/")
print(sql)
print()
duck.execute(sql).df()


In [ ]:
# Count orders per partition year to confirm all years are present.
duck.execute(f"""
    SELECT year, COUNT(*) AS orders
    FROM read_parquet('{PARQUET_DIR}/orders_partitioned/**/*.parquet',
                      hive_partitioning = true)
    GROUP BY year
    ORDER BY year
""").df()


---

## 4 · Min/Max Pruning (Predicate Pushdown)

Parquet stores **min and max statistics** for every column in every row group
(up to ~122,880 rows per group) in the file footer.  When a query filter value
falls outside `[min, max]` for a row group, DuckDB skips that row group
entirely — no decompression, no memory allocation, no I/O for that chunk.

This is **min/max pruning** (also called zone maps).  It works best when the
file is **sorted** on the filter column — sorted files have narrow, non-
overlapping min/max ranges per row group, maximising pruning.

Compare:
- **Sorted file** — filter `l_shipdate = '1998-09-01'` → only 1-2 row groups touched
- **Unsorted file** — every row group contains dates from 1992-1998 → all groups read

> **Bloom filters** complement min/max: they help equality predicates on
> unsorted high-cardinality columns (e.g. `WHERE l_orderkey = 12345`) where
> min/max would still include the row group but the key is absent.


In [ ]:
# EXPLAIN shows the query plan including Parquet statistics usage.
# Look for 'Parquet Files' and 'Row Groups Read' in the output.
sql = load_section("02_partitioning_and_pushdown.sql", "minmax_pruning")
sql = sql.replace("'data/parquet/lineitem.parquet'",
                  f"'{PARQUET_DIR}/lineitem.parquet'")
print(sql)
print()
explain_df = duck.execute(sql).df()
explain_df


In [ ]:
import time

lineitem_path = f"{PARQUET_DIR}/lineitem.parquet"

# Query on a selective date — DuckDB can skip row groups outside this date.
selective_sql = f"""
    SELECT COUNT(*), SUM(l_extendedprice)
    FROM read_parquet('{lineitem_path}')
    WHERE l_shipdate = DATE '1998-09-01'
"""

# Full scan — no filter, all row groups read.
full_sql = f"""
    SELECT COUNT(*), SUM(l_extendedprice)
    FROM read_parquet('{lineitem_path}')
"""

# Warm up (DuckDB caches OS pages after first read)
duck.execute(selective_sql)
duck.execute(full_sql)

runs = 3
selective_times, full_times = [], []
for _ in range(runs):
    t0 = time.perf_counter()
    duck.execute(selective_sql)
    selective_times.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    duck.execute(full_sql)
    full_times.append(time.perf_counter() - t0)

import statistics
print(f"Selective filter (l_shipdate = '1998-09-01'): {statistics.median(selective_times)*1000:.1f} ms")
print(f"Full scan (no filter):                        {statistics.median(full_times)*1000:.1f} ms")


---

## 5 · Writing Results Back to Parquet

`COPY (SELECT ...) TO 'file.parquet'` writes any query result to Parquet.
There is no intermediate Python object — the result is streamed directly from
DuckDB's execution engine to the file.

Use `COMPRESSION 'zstd'` for better compression ratios than the default Snappy,
at similar read speeds.  The output file is valid Parquet readable by Spark,
pandas, DuckDB, and any other Parquet-aware tool.


In [ ]:
out_path = PARQUET_DIR / "nation_revenue_by_year.parquet"

sql = load_section("02_partitioning_and_pushdown.sql", "write_parquet")
sql = (sql
       .replace("'data/parquet/orders.parquet'",   f"'{PARQUET_DIR}/orders.parquet'")
       .replace("'data/parquet/customer.parquet'",  f"'{PARQUET_DIR}/customer.parquet'")
       .replace("'data/parquet/nation.parquet'",    f"'{PARQUET_DIR}/nation.parquet'")
       .replace("'data/parquet/nation_revenue_by_year.parquet'",
                f"'{out_path}'"))

print(sql)
print()
duck.execute(sql)
size_kb = out_path.stat().st_size / 1024
print(f"Written: {out_path.name}  ({size_kb:.1f} KB)")


In [ ]:
# Read the output file back and verify
duck.execute(f"""
    SELECT *
    FROM read_parquet('{out_path}')
    ORDER BY total_revenue DESC
    LIMIT 10
""").df()


---

## Summary

| Feature | How |
|:---|:---|
| Query a file directly | `FROM read_parquet('file.parquet')` |
| Query many files | Glob: `'dir/**/*.parquet'` |
| Hive partition pruning | `hive_partitioning = true` + directory names `col=value/` |
| Min/max row group pruning | Automatic — DuckDB reads Parquet footer statistics |
| Schema inference | `DESCRIBE SELECT * FROM read_parquet(...)` |
| Write results | `COPY (SELECT ...) TO 'out.parquet' (FORMAT PARQUET, COMPRESSION 'zstd')` |

**Pruning hierarchy (coarse to fine):**
1. **Partition pruning** — skip entire files/directories via Hive layout
2. **Min/max pruning** — skip row groups via footer statistics (range predicates)
3. **Bloom filter** — skip row groups for equality predicates on unsorted columns

**Next:** [Notebook 07 — Benchmarking: PostgreSQL vs DuckDB](07_benchmarking.ipynb)
